# MovieLens 25M Data Acquisition

**This notebook:**
- Loads all six MovieLens 25M CSV files into DuckDB
- Converts them to Parquet format
- Verifies that all tables loaded correctly

**Note:** The original CSV files total over 1 GB. Parquet is used for the final dataset because it is more efficient for analytical queries.

In [7]:
# Imports
import duckdb
import logging
import os

# Configure logging to write acquisition progress to a log file
logging.basicConfig(
    filename='data_acquisition.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info('Data acquisition started')

# Path to raw CSV files downloaded from grouplens.org/datasets/movielens/25m/
DATA_PATH = '/Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m/'
PARQUET_PATH = '/Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/'

# Create Parquet output folder if it does not exist
os.makedirs(PARQUET_PATH, exist_ok=True)

print('Setup complete.')
logging.info('Setup complete')

Setup complete.


In [9]:
# Connect to a persistent DuckDB database file stored in OneDrive
# This means pipeline.ipynb can connect to the same file without reloading data
con = duckdb.connect()

# Map table names to their source CSV filenames
tables = {
    'ratings': 'ratings.csv',
    'movies': 'movies.csv',
    'tags': 'tags.csv',
    'genome_scores': 'genome-scores.csv',
    'genome_tags': 'genome-tags.csv',
    'links': 'links.csv'
}

# Load each CSV into DuckDB and write to Parquet
try:
    for table_name, filename in tables.items():
        csv_path = f'{DATA_PATH}{filename}'
        parquet_path = f'{PARQUET_PATH}{table_name}.parquet'

        # Drop table if it already exists so this cell can be rerun safely
        con.execute(f'DROP TABLE IF EXISTS {table_name}')

        # Read CSV and create table in DuckDB
        con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM read_csv_auto('{csv_path}')")

        # Write table to Parquet format for efficient storage and sharing
        con.execute(f"COPY {table_name} TO '{parquet_path}' (FORMAT PARQUET)")

        row_count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        print(f'{table_name}: {row_count:,} rows -> saved to {parquet_path}')
        logging.info(f'{table_name}: {row_count} rows loaded and converted to parquet')

    print('\nAll tables loaded and converted to Parquet successfully.')
    logging.info('All tables loaded and converted to Parquet successfully')

except Exception as e:
    logging.error(f'Error loading data: {e}')
    raise

ratings: 25,000,095 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/ratings.parquet
movies: 62,423 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/movies.parquet
tags: 1,093,360 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/tags.parquet
genome_scores: 15,584,448 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/genome_scores.parquet
genome_tags: 1,128 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/genome_tags.parquet
links: 62,423 rows -> saved to /Users/margauxreynolds/Library/CloudStorage/OneDrive-UniversityofVirginia/ds4320-project1/ml-25m-parquet/links.parquet

All tables loaded and converted to Parquet successfully.


In [10]:
# Verify all tables are present and row counts match expectations
try:
    for table_name in tables.keys():
        count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        print(f'{table_name}: {count:,} rows')
        logging.info(f'Verified {table_name}: {count} rows')

    print('\nAll tables verified. Database ready for pipeline.ipynb.')
    logging.info('Data acquisition complete')

except Exception as e:
    logging.error(f'Error verifying tables: {e}')
    raise

ratings: 25,000,095 rows
movies: 62,423 rows
tags: 1,093,360 rows
genome_scores: 15,584,448 rows
genome_tags: 1,128 rows
links: 62,423 rows

All tables verified. Database ready for pipeline.ipynb.
